**Chess Quant Analytics Project**

Analyzed 200 rated chess games using metrics such as:

*   Volatility
*   Drawdown
*   Blunder Frequency
*   Time Pressure

to identify factors associated with game outcomes.

In [2]:
import pandas as pd

all_games = []

**Data Loading**

In [3]:
df = pd.read_excel("QuantProjectData.xlsx")

**Summary Metric Generation**

In [ ]:
all_games = []

for game_id in df['GameID'].unique():

    df_game = df[df['GameID'] == game_id].copy()

    df_game['Eval'] = pd.to_numeric(df_game['Eval'])
    df_game['EvalChange'] = pd.to_numeric(df_game['EvalChange'])
    df_game['TimeSpent'] = pd.to_numeric(df_game['TimeSpent'])

    df_playable = df_game[
        (df_game['Eval'].abs() < 999) &
        (df_game['EvalChange'].abs() < 100)
    ]

    my_color = df_game['MyColor'].iloc[0]

    if my_color == "White":
        drawdown = df_game['EvalChange'].min()
    else:
        drawdown = df_game['EvalChange'].max()

    volatility = df_playable['EvalChange'].std()

    blunders = (df_game['Blunder?'] == "Yes").sum()
    mistakes = (df_game['Mistake?'] == "Yes").sum()
    inaccuracies = (df_game['Inaccuracy?'] == "Yes").sum()

    avg_time = df_game['TimeSpent'].mean()

    correlation = df_game['TimeSpent'].corr(df_game['EvalChange'])

    game_summary = {
        "GameID": game_id,
        "MyColor": my_color,
        "Volatility": volatility,
        "Drawdown": drawdown,
        "Blunders": blunders,
        "Mistakes": mistakes,
        "Inaccuracies": inaccuracies,
        "AvgTimeSpent": avg_time,
        "TimePressureCorrelation": correlation,
        "Result": df_game['Result'].iloc[0],
        "Opening": df_game['Opening'].iloc[0],
        "MyRating": df_game['MyRating'].iloc[0],
        "OpponentRating": df_game['OpponentRating'].iloc[0],
        "TimeControl": df_game['TimeControl'].iloc[0]
    }

    all_games.append(game_summary)

summary_df = pd.DataFrame(all_games)

summary_df

**Statistical Analysis**

In [ ]:
summary_df.describe()

In [ ]:
summary_df.groupby('Result')['Volatility'].mean()

In [ ]:
summary_df[['AvgTimeSpent','Blunders']].corr()

In [ ]:
summary_df[['AvgTimeSpent','Volatility']].corr()

In [ ]:
summary_df.groupby('MyColor')[['Volatility','Blunders','Drawdown']].mean()

In [ ]:
summary_df[['Result','Volatility']].sort_values(
    by='Volatility',
    ascending=False
).head(15)


In [ ]:
summary_df.groupby('Opening').agg({
    'Volatility':'mean',
    'Blunders':'mean'
}).sort_values('Volatility', ascending=False)

In [ ]:
summary_df.groupby('Result')[['Blunders','Mistakes','Inaccuracies']].mean()

In [ ]:
opening_counts = summary_df['Opening'].value_counts()
opening_counts.head(20)

In [ ]:
summary_df['RatingDiff'] = (
    summary_df['MyRating']
    - summary_df['OpponentRating']
)

summary_df[['RatingDiff','Blunders']].corr()

In [ ]:
summary_df[['RatingDiff','Volatility']].corr()

In [ ]:
opening_stats = summary_df.groupby('Opening').agg({
    'Volatility':'mean',
    'Blunders':'mean',
    'GameID':'count'
})

opening_stats.columns = ['Volatility','Blunders','Games']

opening_stats[opening_stats['Games'] >= 5]

In [ ]:
summary_df[['Blunders',
            'Volatility',
            'AvgTimeSpent',
            'RatingDiff']].corr()

**Visualizations**

In [ ]:
import matplotlib.pyplot as plt

blunder_by_result = summary_df.groupby('Result')['Blunders'].mean()

plt.figure(figsize=(6,4))
bars = plt.bar(
    blunder_by_result.index,
    blunder_by_result.values
)

plt.title("Average Blunders by Game Result")
plt.xlabel("Result")
plt.ylabel("Average Blunders")

for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y + 0.03,
        f"{y:.2f}",
        ha='center'
    )

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.scatter(
    summary_df['Volatility'],
    summary_df['Blunders']
)

plt.xlabel("Volatility")
plt.ylabel("Blunders")
plt.title("Volatility vs Blunders")

plt.show()

**Key Findings**



*   Blunder frequency was the strongest predictor of result.

*   Average thinking time had almost no correlation with blunders.
*   Higher volatility positions generated more errors.

